In [ ]:
!pip install gdown -q
!pip install split-folders
!pip install tensorflow==2.19.0
!pip install matplotlib==3.9.2
!pip install numpy==2.0
!pip install scipy==1.14.1
!pip install scikit-learn==1.5.2
!pip install -q "ml_dtypes>=0.5.0"

## Download the input data file

In [10]:
import os
import zipfile
import gdown

# Define dataset URL and paths
file_id = "1HY71FfYNGX1lBB9hnfUjL_h9GWLN_w2Q"
local_zip = "waste-dataset.zip"
extract_dir = "waste-dataset"

def download_dataset(file_id, output_file):
    """Download the dataset using wget in quiet mode."""
    print("Downloading the dataset...")
    gdown.download(id=file_id, output=output_file, quiet=False)
    print("Download completed.")

def extract_zip_in_chunks(zip_file, extract_to, batch_size=2000):
    """
    Extract a large zip file in chunks to avoid memory bottlenecks.
    Processes a specified number of files (batch_size) at a time.
    """
    print("Extracting the dataset in chunks...")
    os.makedirs(extract_to, exist_ok=True)  # Ensure the extraction directory exists
    
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        files = zip_ref.namelist()  # List all files in the archive
        total_files = len(files)
        
        for i in range(0, total_files, batch_size):
            batch = files[i:i+batch_size]
            for file in batch:
                zip_ref.extract(file, extract_to)  # Extract each file in the batch
            print(f"Extracted {min(i+batch_size, total_files)} of {total_files} files...")
    
    print(f"Dataset successfully extracted to '{extract_to}'.")

# Main script execution
if __name__ == "__main__":

    os.system("pip install gdown -q")

    # Download the dataset if not already downloaded
    if not os.path.exists(local_zip):
        download_dataset(file_id, local_zip)
    else:
        print("Dataset already downloaded.")
    
    # Extract the dataset if not already extracted
    if not os.path.exists(extract_dir):
        extract_zip_in_chunks(local_zip, extract_dir)
    else:
        print("Dataset already extracted.")
    
    # Optional cleanup of the zip file
    if os.path.exists(local_zip):
        os.remove(local_zip)
        print(f"Cleaned up zip file: {local_zip}") 

Downloading...
From (original): https://drive.google.com/uc?id=1HY71FfYNGX1lBB9hnfUjL_h9GWLN_w2Q
From (redirected): https://drive.google.com/uc?id=1HY71FfYNGX1lBB9hnfUjL_h9GWLN_w2Q&confirm=t&uuid=98856462-b164-40e9-8ae7-2170d2b0e56a
To: c:\Users\Jocel\Downloads\AI project\final-project-tensorflow\waste-dataset.zip
100%|██████████| 587M/587M [00:10<00:00, 53.6MB/s] 


Download completed.
Extracting the dataset in chunks...
Extracted 2000 of 5989 files...
Extracted 4000 of 5989 files...
Extracted 5989 of 5989 files...
Dataset successfully extracted to 'waste-dataset'.
Cleaned up zip file: waste-dataset.zip


## Task 1: Import necessary libraries and set dataset paths 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import splitfolders
import tensorflow as tf
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Print TensorFlow version
print(tf.__version__)

# Aplanar el dataset - mover imágenes al nivel de categoría principal
for category in os.listdir("waste-dataset"):
    category_path = os.path.join("waste-dataset", category)
    if os.path.isdir(category_path):
        for subcategory in os.listdir(category_path):
            subcategory_path = os.path.join(category_path, subcategory)
            if os.path.isdir(subcategory_path):
                for img in os.listdir(subcategory_path):
                    src = os.path.join(subcategory_path, img)
                    dst = os.path.join(category_path, f"{subcategory}_{img}")  # prefijo para evitar nombres duplicados
                    shutil.move(src, dst)
                os.rmdir(subcategory_path)

# Eliminar waste-split anterior si existe
if os.path.exists("waste-split"):
    shutil.rmtree("waste-split")

# Split 70% train, 15% val, 15% test
splitfolders.ratio("waste-dataset", output="waste-split", ratio=(0.7, 0.15, 0.15), seed=42)

# Set dataset paths
train_dir = 'waste-split/train'
val_dir   = 'waste-split/val'
test_dir  = 'waste-split/test'

2.16.2


Copying files: 0 files [00:00, ? files/s]


## Task 2: Set up data generators for training, validation, and testing with augmentation 

In [ ]:
# Image data generators
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Load images from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

# Print the length of the training data generator
print(len(train_generator))

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

Found 0 images belonging to 4 classes.
Found 0 images belonging to 4 classes.
Found 0 images belonging to 4 classes.


## Task 3: Define the VGG16-based model architecture with custom layers

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, BatchNormalization, Dropout

# Load VGG16 with pre-trained weights
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Build the model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(train_generator.num_classes, activation='softmax')
])


## Task 4: Compile the model with appropriate loss and optimizer

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Print the summary of the model
model.summary()

## Task 5: Train the model with early stopping and learning rate scheduling

In [ ]:
from tensorflow.keras.mixed_precision import set_global_policy

# Define callbacks
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Enable mixed precision (if on GPU)
set_global_policy('float32')

steps_per_epoch = 50 
validation_steps = 25

history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator,  
    steps_per_epoch=steps_per_epoch,  
    validation_steps=validation_steps,  
    callbacks=[lr_scheduler, early_stopping]
)

## Task 6: Fine-tune the model by unfreezing specific layers in VGG16

In [ ]:
import tensorflow as tf  # Import TensorFlow for accessing tf.keras

# Check the number of layers in the base model
num_layers = len(base_model.layers)
print(f"The base model has {num_layers} layers.")

# Unfreeze the last 5 layers for fine-tuning
for layer in base_model.layers[-5:]:
    layer.trainable = True

# Freeze BatchNorm layers to speed up fine-tuning
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Re-compile the model with a faster optimizer
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),   # Higher learning rate for faster convergence
    metrics=['accuracy']
)

# Continue training with fewer steps per epoch
history_fine = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator,
    steps_per_epoch=steps_per_epoch,  # Reduced steps per epoch
    validation_steps=validation_steps,  # Reduced validation steps
    callbacks=[lr_scheduler, early_stopping]
)

## Task 7: Evaluate the model on the test set and display accuracy

In [ ]:
# Evaluate on the test set
test_loss, test_accuracy = model.evaluate(test_generator, steps=50)
print(f"Test Accuracy: {test_accuracy:.2f}")

## Task 8: Visualize training performance with accuracy and loss curves

In [ ]:
# Plot accuracy and validity curves (Extract Features Model)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Extract Features Model - Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot loss and validity curves (Fine Tune Model)
plt.plot(history_fine.history['loss'], label='Training Loss')
plt.plot(history_fine.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Fine Tune Model - Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot accuracy and validity curves (Fine Tune Model)
plt.plot(history_fine.history['accuracy'], label='Training Accuracy')
plt.plot(history_fine.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Fine Tune Model - Accuracy')
plt.legend()
plt.grid(True)
plt.show()

## Task 9: Test model predictions on sample images and visualize results

In [ ]:
import os
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

# Initialize counters for actual and predicted classes
actual_count = Counter()
predicted_count = Counter()

# Function to get class name from predicted index
def get_class_name_from_index(predicted_index, class_index_mapping):
    """Convert predicted index to class name."""
    for class_name, index in class_index_mapping.items():
        if index == predicted_index:
            return class_name
    return "Unknown"  # Default if index is not found

# Define the function for visualization
def visualize_prediction_with_actual(img_path, class_index_mapping):
    # Extract the true label dynamically from the directory structure
    class_name = os.path.basename(os.path.dirname(img_path))  # Extract folder name (class)
    
    # Load and preprocess the image
    img = load_img(img_path, target_size=(64, 64)) 
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict the class
    prediction = model.predict(img_array)
    predicted_index = np.argmax(prediction, axis=-1)[0]
    predicted_class_name = get_class_name_from_index(predicted_index, class_index_mapping)

    # Update the counters
    actual_count[class_name] += 1
    predicted_count[predicted_class_name] += 1

    # Visualize the image with predictions
    plt.figure(figsize=(2, 2), dpi=100)
    plt.imshow(img)
    plt.title(f"Actual: {class_name}, Predicted: {predicted_class_name}")
    plt.axis('off')
    plt.show()

# Retrieve class index mapping from the training generator
class_index_mapping = train_generator.class_indices
print("Class Index Mapping:", class_index_mapping)  # Debugging: Check the mapping

# Define a list of image paths
sample_images = []
for category in os.listdir(test_dir):
    category_path = os.path.join(test_dir, category)
    if os.path.isdir(category_path):
        images = os.listdir(category_path)
        if images:
            sample_images.append(os.path.join(category_path, images[0]))

# Run the predictions and visualization
for img_path in sample_images:
    visualize_prediction_with_actual(img_path, class_index_mapping)